# ArcVox — GPU Engine Verification (Colab / Kaggle)

**Purpose:** prove the *pretrained, open-source* engines ArcVox ships with produce
commercial-grade output — on a free GPU, with **no model training**. Includes a
dedicated **Indian-languages** section (Tamil, Telugu, Hindi, Bengali, …).

> You are **not** training anything. These weights were already trained by
> Resemble AI (Chatterbox), AI4Bharat (Indic Parler-TTS), OpenAI/SYSTRAN
> (Whisper) and the avatar labs. We download them and run inference — exactly
> what the ArcVox backend does.

Artifacts you'll produce:
1. **English + multilingual TTS + zero-shot clone** (Chatterbox) → `.wav`
2. **Indian-language TTS** (AI4Bharat Indic Parler-TTS, Apache-2.0) → `.wav` in Tamil/Telugu/Hindi
3. **Transcription** of all of the above (faster-whisper `large-v3`)
4. **Talking-head avatar** (SadTalker) → `.mp4`

> ⚠️ **Privacy:** running here uploads samples to Google/Kaggle infra — fine for
> *your own testing*, but **not** the private production deployment.


## 0 · Confirm GPU
Runtime → Change runtime type → **GPU** (Colab) / enable the GPU accelerator (Kaggle).

In [ ]:
!nvidia-smi

In [ ]:
# Cross-platform upload helper (Colab + Kaggle).
import os

def upload_one(prompt="Upload a file"):
    try:
        from google.colab import files
        print(prompt)
        up = files.upload()
        return list(up.keys())[0]
    except Exception:
        print("Kaggle (or no Colab uploader): add the file via 'Add Data'/'Upload',")
        print("then set the path manually, e.g.  ref = '/kaggle/input/my-voice/sample.wav'")
        return None

IS_KAGGLE = os.path.exists("/kaggle")
print("Environment:", "Kaggle" if IS_KAGGLE else "Colab / other")

## 1 · Voice — Chatterbox (English + multilingual + zero-shot clone)

Chatterbox (MIT, Resemble AI) is the HD voice + cloning engine. It's **multilingual
(23 languages incl. Hindi)** and clones any voice from a short sample — no training,
just a reference clip at inference time. Same engine as `backend/engines/tts.py`.

In [ ]:
!pip install -q chatterbox-tts

In [ ]:
import torch, torchaudio as ta
from chatterbox.tts import ChatterboxTTS
from IPython.display import Audio, display

cb = ChatterboxTTS.from_pretrained(device="cuda")

# English
wav = cb.generate("This is ArcVox, running entirely on a GPU you control.")
ta.save("arcvox_tts.wav", wav, cb.sr)
display(Audio("arcvox_tts.wav"))

# Hindi (Chatterbox supports it natively)
hi = cb.generate("नमस्ते, यह आपके अपने हार्डवेयर पर चल रहा है।")
ta.save("arcvox_hindi_cb.wav", hi, cb.sr)
display(Audio("arcvox_hindi_cb.wav"))

In [ ]:
# Zero-shot voice clone: upload a clean 10-30s sample of ONE speaker.
ref = upload_one("Upload a 10-30s voice sample (wav/mp3) to clone:")
if ref:
    cloned = cb.generate("Now I am speaking in the cloned voice. Your hardware, your data.",
                         audio_prompt_path=ref)
    ta.save("arcvox_clone.wav", cloned, cb.sr)
    display(Audio("arcvox_clone.wav"))
else:
    print("No reference set — skipping clone. Set `ref` and re-run.")

In [ ]:
# Free Chatterbox before the next model (16GB can't hold everything at once).
import gc
del cb; gc.collect(); torch.cuda.empty_cache()
print("Chatterbox unloaded.")

## 1b · Indian languages — AI4Bharat Indic Parler-TTS  *(21 languages)*

**`ai4bharat/indic-parler-tts`** — Apache-2.0 (commercial-safe), covers **Hindi,
Tamil, Telugu, Bengali, Gujarati, Kannada, Malayalam, Marathi, Punjabi, Odia,
Assamese, Urdu, Kashmiri, Sanskrit, Sindhi, Nepali + English**. The voice is set
by a **natural-language description**; the language comes from the input **script**.
Same engine wired into the backend as `TTS_ENGINE=indic_parler`.

> 🔒 **Gated model.** Accept the terms once at
> https://hf.co/ai4bharat/indic-parler-tts, then paste a token from
> https://hf.co/settings/tokens below.

In [ ]:
# Parler-TTS library + HF auth (the model is gated).
!pip install -q git+https://github.com/huggingface/parler-tts.git
from huggingface_hub import login
login()  # paste your HF token (needs access to ai4bharat/indic-parler-tts)

In [ ]:
import torch, soundfile as sf
from parler_tts import ParlerTTSForConditionalGeneration
from transformers import AutoTokenizer
from IPython.display import Audio, display

MODEL = "ai4bharat/indic-parler-tts"
ip = ParlerTTSForConditionalGeneration.from_pretrained(MODEL).to("cuda")
prompt_tok = AutoTokenizer.from_pretrained(MODEL)
desc_tok = AutoTokenizer.from_pretrained(ip.config.text_encoder._name_or_path)

def indic_say(text, description, out):
    d = desc_tok(description, return_tensors="pt").to("cuda")
    p = prompt_tok(text, return_tensors="pt").to("cuda")
    gen = ip.generate(input_ids=d.input_ids, attention_mask=d.attention_mask,
                      prompt_input_ids=p.input_ids, prompt_attention_mask=p.attention_mask)
    sf.write(out, gen.cpu().numpy().squeeze(), ip.config.sampling_rate)
    return out

DESC = "A clear, neutral narrator with crisp, studio-quality recording and a moderate pace."

# Tamil, Telugu, Hindi — type in the native script and the model speaks it.
display(Audio(indic_say("வணக்கம், இது உங்கள் சொந்த சர்வரில் இயங்குகிறது.", DESC, "tamil.wav")))
display(Audio(indic_say("నమస్తే, ఇది మీ స్వంత హార్డ్‌వేర్‌లో నడుస్తోంది.", DESC, "telugu.wav")))
display(Audio(indic_say("नमस्ते, यह पूरी तरह आपके सर्वर पर चल रहा है।", DESC, "hindi.wav")))
print("Saved tamil.wav, telugu.wav, hindi.wav")

In [ ]:
# Free Indic Parler before Whisper.
import gc
del ip; gc.collect(); torch.cuda.empty_cache()
print("Indic Parler-TTS unloaded.")

## 2 · Transcription — faster-whisper `large-v3`

The strongest module: `large-v3` matches or beats commercial APIs and covers
**Indian languages too** (Hindi, Tamil, Telugu, Bengali, …). We transcribe both
an English clip and one of the Indian-language clips we just generated. Same
engine as `backend/engines/stt.py`.

In [ ]:
!pip install -q faster-whisper

In [ ]:
from faster_whisper import WhisperModel
asr = WhisperModel("large-v3", device="cuda", compute_type="float16")

def show_transcript(path):
    if not os.path.exists(path):
        print("(skip)", path); return
    segs, info = asr.transcribe(path, vad_filter=True)
    print(f"\n=== {path}  ·  lang={info.language} (p={info.language_probability:.2f}) ===")
    for s in segs:
        print(f"[{s.start:5.1f}-{s.end:5.1f}] {s.text.strip()}")

for p in ["arcvox_tts.wav", "arcvox_clone.wav", "tamil.wav", "telugu.wav", "hindi.wav"]:
    show_transcript(p)

In [ ]:
import gc
del asr; gc.collect(); torch.cuda.empty_cache()
print("Whisper unloaded.")

## 3 · Talking-head avatar — SadTalker  *(heavier, optional)*

Lip-syncs **one portrait photo** to audio (~8GB VRAM). Most fragile install
(specific torch/ffmpeg + weight downloads), so it's last. If a cell errors it's
usually a dependency pin — the voice/transcription verification above stands alone.
Backend alternatives: MuseTalk, EchoMimic, LivePortrait.

In [ ]:
%cd /content 2>/dev/null || cd /kaggle/working
!git clone https://github.com/OpenTalker/SadTalker
%cd SadTalker
!pip install -q -r requirements.txt

In [ ]:
!bash scripts/download_models.sh

In [ ]:
photo = upload_one("Upload a clear, front-facing portrait photo:")
audio_path = None
for cand in ["/content/hindi.wav", "/content/arcvox_clone.wav", "/content/arcvox_tts.wav",
             "/kaggle/working/hindi.wav", "/kaggle/working/arcvox_tts.wav"]:
    if os.path.exists(cand):
        audio_path = cand; break
print("Audio:", audio_path, "| Photo:", photo)

In [ ]:
!python inference.py \
    --driven_audio "$audio_path" \
    --source_image "$photo" \
    --result_dir ./results \
    --still --preprocess full --enhancer gfpgan

In [ ]:
import glob
from IPython.display import Video
vids = sorted(glob.glob("./results/**/*.mp4", recursive=True), key=os.path.getmtime)
print("Generated:", vids[-1] if vids else "none")
Video(vids[-1], embed=True) if vids else print("No video — check the log above.")

## What this proves

- ✅ **No training** — pretrained MIT/Apache weights, inference only.
- ✅ **English + multilingual voice** (Chatterbox) and **21-language Indian TTS**
  (Indic Parler-TTS, Apache-2.0) both produce real audio.
- ✅ **Transcription** is commercial-grade across English *and* Indian languages.
- ✅ **Avatars** work (honest: good, still a step behind HeyGen).

**Wiring it into the product:** set `TTS_ENGINE=indic_parler` (+ `HF_TOKEN`) for an
India-focused deployment, or `TTS_ENGINE=chatterbox` for English/global + cloning.
The TTS Studio shows a language picker automatically when an Indic engine is active.
